**© Copyright AIDENTIFY. All rights reserved.**

# Part 5 | Session 39: 프로젝트 데이터 파이프라인 구축

## 🎯 데이터 파이프라인 구축

이 노트북에서는 프로젝트에 필요한 **데이터 파이프라인 전체**를 구축합니다.

### 파이프라인 흐름
```
데이터 수집 → 정제 → 형식 변환 → 합성 데이터 생성 → 검증 → Train/Eval 분할
```

### 사전 준비
- `project_plan.json` — **이 노트북 0️⃣ 단계에서 생성**합니다 (별도 사전 노트북 불필요)
- OpenAI API 키 (합성 데이터 생성 시)
- HuggingFace 토큰 (데이터셋 다운로드 시)

---
## 0️⃣ 프로젝트 계획 정의 (`project_plan.json` 생성)

이 파이프라인은 **무엇을 · 얼마나 · 어떤 톤으로** 데이터를 만들지 담은 `project_plan.json` 을 입력으로 사용합니다.
별도 사전 노트북 없이, 아래 셀에서 계획서를 정의하고 저장합니다. (도메인 / 샘플 수 / 토픽은 자유롭게 수정하세요.)

In [ ]:
# =====================================================
# 0️⃣ 프로젝트 계획 정의 및 저장 (project_plan.json)
#    - 39 를 자급자족으로 실행하기 위한 첫 단계
#    - 40 / 41 / 42 가 그대로 읽는 계획서 (task, eval_prompts 포함)
# =====================================================
import os, json

# ── 다운스트림(40/41/42)과 동일한 경로 규칙 ──
PLAN_PATH = "project_plan.json"     # 40/41/42 가 현재 폴더에서 읽음
DATA_DIR  = "outputs/data"          # 40: train.json / 41: 참조답변
os.makedirs(DATA_DIR, exist_ok=True)

project_plan = {
    "project_name": "의료 상담 어시스턴트",
    "domain": "의료 상담",
    "task": "사용자의 의료 관련 질문에 정확하고 안전하게 답변하는 상담 어시스턴트",
    "data_config": {
        "num_samples": 12,               # 목표 데이터 수 (빠른 실행용 소량; 늘리려면 조정 — 예: 60)
        "seed_topics": [
            "고혈압 관리와 생활습관",
            "당뇨병 초기 증상과 예방",
            "감기와 독감의 차이",
            "건강검진 결과 해석",
            "일반 의약품 복용 주의사항",
            "예방접종 권장 일정",
        ],
        "system_prompt": (
            "당신은 신뢰할 수 있는 의료 정보를 제공하는 상담 어시스턴트입니다. "
            "정확하고 이해하기 쉽게 설명하되, 진단·처방은 하지 않고 "
            "필요하면 전문의 상담을 권유합니다."
        ),
    },
    # 41(평가)·42(배포) 가 사용하는 평가용 프롬프트
    "eval_prompts": [
        "고혈압을 관리하려면 생활습관을 어떻게 바꿔야 하나요?",
        "당뇨병 초기에 나타나는 증상은 무엇인가요?",
        "감기와 독감은 어떻게 다른가요?",
        "건강검진에서 콜레스테롤 수치가 높으면 어떻게 해야 하나요?",
        "해열제를 복용할 때 주의할 점은 무엇인가요?",
    ],
}

with open(PLAN_PATH, "w", encoding="utf-8") as f:
    json.dump(project_plan, f, ensure_ascii=False, indent=2)

print(f"✅ 프로젝트 계획 저장: {PLAN_PATH}")
print(f"   도메인={project_plan['domain']}, "
      f"목표={project_plan['data_config']['num_samples']}건, "
      f"토픽={len(project_plan['data_config']['seed_topics'])}개, "
      f"평가프롬프트={len(project_plan['eval_prompts'])}개")


In [ ]:
# =====================================================
# 📦 라이브러리 임포트 + 프로젝트 계획 로드
# =====================================================
import json, os, time, requests
from dotenv import load_dotenv
load_dotenv("../.env")  # OPENAI_API_KEY 로드

# 계획서는 0️⃣ 단계에서 현재 폴더에 저장됨 (40/41/42 도 여기서 읽음)
PLAN_PATH = "project_plan.json"
with open(PLAN_PATH, "r", encoding="utf-8") as f:
    project_plan = json.load(f)

# OpenAI API key 확인 → 없으면 Ollama 사용
USE_OPENAI = False
try:
    from openai import OpenAI
    client = OpenAI()
    # key 유효성 테스트
    client.models.list()
    USE_OPENAI = True
    print("✅ OpenAI API 사용 (gpt-4o-mini)")
except Exception:
    print("⚠️ OpenAI API key가 없거나 유효하지 않습니다.")
    # Ollama 확인
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            print("✅ Ollama 사용 (로컬 모델)")
        else:
            print("❌ Ollama도 실행 중이 아닙니다. ollama serve를 실행하세요.")
    except:
        print("❌ Ollama도 실행 중이 아닙니다. ollama serve를 실행하세요.")

print(f"\n📋 프로젝트: {project_plan['project_name']}")
print(f"📌 도메인: {project_plan['domain']}")
print(f"📌 생성할 데이터: {project_plan['data_config']['num_samples']}건")


In [ ]:
# =====================================================
# 🤖 학습 데이터 자동 생성 (OpenAI 또는 Ollama)
# =====================================================
domain = project_plan["domain"]
num_samples = project_plan["data_config"]["num_samples"]
seed_topics = project_plan["data_config"]["seed_topics"]
samples_per_topic = num_samples // len(seed_topics)

print(f"🚀 {domain} 도메인 데이터 {num_samples}건 생성 시작!")
print(f"   엔진: {'OpenAI gpt-4o-mini' if USE_OPENAI else 'Ollama (로컬)'}")
print(f"   토픽 {len(seed_topics)}개 x 토픽당 {samples_per_topic}건")
print("="*60)

def generate_with_openai(topic, n):
    """OpenAI API로 데이터 생성"""
    prompt = f"""당신은 '{domain}' 분야의 교육 데이터를 생성하는 전문가입니다.

주제: {topic}

이 주제에 대해 {n}개의 instruction-output 쌍을 JSON 배열로 생성하세요.

규칙:
- instruction: 학습자가 물어볼 법한 자연스러운 한국어 질문
- output: 200~400자 분량의 정확하고 상세한 한국어 답변
- 각 쌍은 서로 다른 관점의 질문이어야 함

반드시 아래 JSON 형식으로만 답변하세요:
[{{"instruction": "질문", "input": "", "output": "답변"}}]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    if isinstance(result, list):
        return result
    elif isinstance(result, dict):
        return list(result.values())[0] if result else []
    return []

def generate_with_ollama(topic, n):
    """Ollama로 데이터 생성"""
    prompt = f"""당신은 '{domain}' 분야의 교육 데이터를 생성하는 전문가입니다.

주제: {topic}

이 주제에 대해 {n}개의 질문-답변 쌍을 만들어주세요.
각 답변은 200~400자로 정확하고 상세하게 작성하세요.

반드시 아래 JSON 배열 형식으로만 답변하세요. 다른 텍스트 없이 JSON만 출력하세요:
[{{"instruction": "질문", "input": "", "output": "답변"}}]"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:1.5b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.8, "num_predict": 2048},
        },
        timeout=120,
    )
    
    if response.status_code != 200:
        return []
    
    text = response.json().get("response", "")
    
    # JSON 추출 시도
    try:
        # 전체가 JSON인 경우
        return json.loads(text)
    except:
        pass
    
    try:
        # JSON 부분만 추출
        start = text.find("[")
        end = text.rfind("]") + 1
        if start >= 0 and end > start:
            return json.loads(text[start:end])
    except:
        pass
    
    return []

# 데이터 생성 실행
generated_data = []
generate_fn = generate_with_openai if USE_OPENAI else generate_with_ollama

for topic_idx, topic in enumerate(seed_topics):
    print(f"\n⏳ [{topic_idx+1}/{len(seed_topics)}] 토픽: {topic}")
    
    try:
        items = generate_fn(topic, samples_per_topic)
        
        for item in items:
            if item.get("instruction") and item.get("output"):
                generated_data.append({
                    "instruction": item["instruction"],
                    "input": item.get("input", ""),
                    "output": item["output"],
                })
        
        print(f"   ✅ {len(items)}건 생성 (누적: {len(generated_data)}건)")
        time.sleep(1)
        
    except Exception as e:
        print(f"   ❌ 에러: {e}")

print(f"\n{'='*60}")
print(f"✅ 데이터 생성 완료: 총 {len(generated_data)}건")

if len(generated_data) < num_samples:
    print(f"⚠️ 목표({num_samples}건)보다 적게 생성됨. 토픽을 추가하거나 다시 실행하세요.")


---
## 1️⃣ 데이터 수집

도메인에 맞는 데이터를 수집합니다. 아래 방법 중 해당하는 것을 선택하세요.

### 수집 방법 선택
| 방법 | 장점 | 단점 | 적합한 경우 |
|------|------|------|------------|
| HuggingFace 데이터셋 | 즉시 사용 가능, 정제됨 | 도메인 제한 | 공개 데이터 있을 때 |
| 웹 크롤링 | 최신 데이터 | 노이즈 많음, 법적 이슈 | 특수 도메인 |
| 합성 데이터 (API) | 품질 제어 가능 | 비용 발생 | 데이터 부족 시 |
| 기존 데이터 변환 | 비용 없음 | 형식 변환 필요 | 자체 데이터 있을 때 |

In [ ]:
# =====================================================
# 📊 생성된 데이터 확인
# =====================================================
print(f"📊 생성된 데이터: {len(generated_data)}건")
print("="*60)

for i, item in enumerate(generated_data[:5]):
    print(f"\n{i+1}. Q: {item['instruction'][:60]}...")
    print(f"   A: {item['output'][:80]}...")

print(f"\n... (총 {len(generated_data)}건)")


---
## 2️⃣ 데이터 정제 파이프라인

수집한 원시 데이터를 정제합니다.

### 정제 단계
1. **중복 제거**: 완전 일치 / 유사도 기반
2. **길이 필터링**: 너무 짧거나 긴 데이터 제거
3. **품질 필터링**: 의미 없는 데이터 제거
4. **언어 필터링**: 한국어 비율 확인
5. **특수문자 정리**: HTML 태그, 불필요한 기호 제거

In [ ]:
# =====================================================
# 🧹 데이터 정제
# =====================================================
print("🧹 데이터 정제 중...")

# 중복 제거
seen = set()
cleaned_data = []
for item in generated_data:
    key = item["instruction"]
    if key not in seen:
        seen.add(key)
        cleaned_data.append(item)

print(f"   중복 제거: {len(generated_data)} → {len(cleaned_data)}건")

# 길이 필터링 (답변 10자 이상)
cleaned_data = [item for item in cleaned_data if len(item["output"]) >= 10]
print(f"   길이 필터: → {len(cleaned_data)}건")

print(f"✅ 정제 완료: {len(cleaned_data)}건")


In [ ]:
# =====================================================
# 🔄 ChatML 형식으로 변환
# =====================================================
system_prompt = project_plan["data_config"]["system_prompt"]

formatted_data = []
for item in cleaned_data:
    instruction = item["instruction"]
    input_text = item.get("input", "")
    output_text = item["output"]
    
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    
    formatted_data.append({
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output_text}
        ]
    })

print(f"✅ ChatML 변환 완료: {len(formatted_data)}건")
print(f"\n--- 변환 예시 ---")
for msg in formatted_data[0]["messages"]:
    print(f"  [{msg['role']}] {msg['content'][:80]}...")


---
## 3️⃣ 데이터 형식 변환

정제된 데이터를 학습에 적합한 형식으로 변환합니다.

### 지원 형식

**Alpaca 형식** (Instruction Tuning 표준)
```json
{"instruction": "질문", "input": "추가 컨텍스트", "output": "답변"}
```

**ShareGPT 형식** (대화형)
```json
{"conversations": [{"from": "human", "value": "질문"}, {"from": "gpt", "value": "답변"}]}
```

In [ ]:
# =====================================================
# 📊 데이터 통계 분석
# =====================================================
print("📊 데이터 통계")
print("="*60)

if not formatted_data:
    print("   ⚠️ 생성된 데이터가 0건입니다 — OpenAI 키(.env) 또는 Ollama 실행을 확인하고 위 생성 셀을 다시 실행하세요.")
else:
    lengths = [len(item["messages"][2]["content"]) for item in formatted_data]
    print(f"   총 데이터: {len(formatted_data)}건")
    print(f"   평균 답변 길이: {sum(lengths)/len(lengths):.0f}자")
    print(f"   최소/최대: {min(lengths)}/{max(lengths)}자")

    for i, item in enumerate(formatted_data[:3]):
        print(f"\n  {i+1}. {item['messages'][1]['content'][:50]}... ({len(item['messages'][2]['content'])}자)")


---
## 4️⃣ 합성 데이터 생성 (OpenAI API 활용)

데이터가 부족한 경우 **OpenAI API를 활용하여 합성 데이터를 생성**합니다.

### 합성 데이터 생성 전략
- **Seed 기반 생성**: 기존 데이터를 참고하여 유사한 새 데이터 생성
- **시나리오 기반 생성**: 도메인 시나리오를 정의하고 Q&A 쌍 생성
- **Distillation**: 큰 모델의 출력을 작은 모델의 학습 데이터로 활용

> 💡 **Hint**: `gpt-4o-mini`를 사용하면 비용을 절약하면서도 좋은 품질의 데이터를 생성할 수 있습니다.

---
## 5️⃣ 데이터 검증 및 통계

최종 데이터의 품질을 검증하고 통계를 확인합니다.

In [ ]:
# =====================================================
# ✅ 데이터 품질 검증
# =====================================================
print("✅ 품질 검증")
print("="*60)

errors = 0
for i, item in enumerate(formatted_data):
    msgs = item["messages"]
    if len(msgs) != 3:
        print(f"  ❌ {i}: 메시지 수 {len(msgs)}")
        errors += 1
    if not msgs[2]["content"].strip():
        print(f"  ❌ {i}: 빈 답변")
        errors += 1

if errors == 0:
    print("  ✅ 모든 데이터 정상!")
else:
    print(f"  ⚠️ {errors}건 문제 발견")


In [ ]:
# =====================================================
# 📊 답변 품질 샘플 확인
# =====================================================
import random
random.seed(42)

samples = random.sample(formatted_data, min(3, len(formatted_data)))
for i, item in enumerate(samples):
    print(f"\n{'='*60}")
    print(f"📌 샘플 {i+1}")
    print(f"Q: {item['messages'][1]['content']}")
    print(f"A: {item['messages'][2]['content'][:300]}")


---
## 6️⃣ Train/Eval 분할

검증된 데이터를 **학습용(Train)**과 **평가용(Eval)**으로 분할합니다.

In [ ]:
# =====================================================
# 📊 Train/Eval 분할 및 저장 (다운스트림 40/41 규칙에 맞춤)
#    - outputs/data/train.json · eval.json : instruction/output 형식 (40 KB, 41 참조답변)
#    - outputs/data/train_chatml.json      : ChatML 형식 (파인튜닝용)
# =====================================================
import os, json, random
random.seed(42)

DATA_DIR = "outputs/data"
os.makedirs(DATA_DIR, exist_ok=True)

# instruction/output 형식 (40 to_doc, 41 참조답변이 읽는 형식)
data_io = [
    {"instruction": it["instruction"], "input": it.get("input", ""), "output": it["output"]}
    for it in cleaned_data
]
random.shuffle(data_io)
split = int(len(data_io) * 0.8)
train_data, eval_data = data_io[:split], data_io[split:]

train_path = os.path.join(DATA_DIR, "train.json")
eval_path  = os.path.join(DATA_DIR, "eval.json")
with open(train_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_data, f, ensure_ascii=False, indent=2)

# ChatML(파인튜닝용)도 함께 저장 — 형식 변환 학습 결과 보존
chatml_path = os.path.join(DATA_DIR, "train_chatml.json")
with open(chatml_path, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print("✅ 데이터 저장 완료!")
print(f"   학습(instruction/output): {len(train_data)}건 → {train_path}")
print(f"   평가(instruction/output): {len(eval_data)}건 → {eval_path}")
print(f"   ChatML(파인튜닝용):        {len(formatted_data)}건 → {chatml_path}")
print(f"\n🎉 다음 단계: Session 40 (40_project_training.ipynb) — 이 train.json 을 KB 로 사용")


---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 데이터 수집 (HuggingFace / 크롤링 / 로컬 파일)
- ✅ 데이터 정제 (중복 제거, 길이 필터, 한국어 필터)
- ✅ 데이터 형식 변환 (Alpaca / ShareGPT)
- ✅ 합성 데이터 생성 (OpenAI API)
- ✅ 데이터 검증 및 통계 분석
- ✅ Train/Eval 분할 및 저장

### 산출물
- 📄 `project_plan.json` - 프로젝트 계획서 (40·41·42 가 읽음)
- 📁 `outputs/data/train.json` - 학습/KB용 데이터 (instruction/output)
- 📁 `outputs/data/eval.json` - 평가용 데이터
- 📁 `outputs/data/train_chatml.json` - 파인튜닝용 ChatML 데이터

### 다음 세션 준비사항
- 📌 데이터 파일이 올바르게 저장되었는지 확인
- 📌 데이터 수가 최소 300개 이상인지 확인
- 📌 데이터 형식이 올바른지 JSON 구조 확인

### 다음 노트북
👉 **Session 40** (`40_project_training.ipynb`): 프로젝트 모델 학습 수행